In [ ]:
pip install agentic-doc -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 104.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.4/323.4 kB 33.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 84.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 313.2/313.2 kB 33.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.9/14.9 MB 109.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 10.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.19.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.3, but you have protobuf 6.33.6 which is incompatible.
google-ai-generativelanguag

In [ ]:


import os
from agentic_doc.parse import parse_documents
from pydantic import BaseModel
from typing import Optional
from collections import Counter

# ── Set API key ───────────────────────────────────────────────────────────────
os.environ["VISION_AGENT_API_KEY"] = "MWd2Mmd5ajNrYmxlZWhpMW5hd28wOlpQdldUaGJYNlZ3anlrM2R2VE9tWERLV3VxSE1YYVNY"  # ← thay vào đây

# ── Parse PDF ─────────────────────────────────────────────────────────────────
print("⏳ Đang parse SAWACO BCTC...")
results = parse_documents(["/kaggle/input/datasets/daotiendao/ccccccc/SAWACO_Baocaotaichinh_Q3_2025_Congtyme.pdf"])
doc = results[0]

# ── Kiểm tra tổng quan ────────────────────────────────────────────────────────
print("\n" + "="*60)
print("📄 TỔNG QUAN KẾT QUẢ PARSE")
print("="*60)
print(f"Loại file   : {doc.doc_type}")
print(f"Tổng chunks : {len(doc.chunks)}")
print(f"Trang       : {doc.start_page_idx} → {doc.end_page_idx}")

# Thống kê chunk types
chunk_types = Counter(c.chunk_type.value for c in doc.chunks)
print("\n📊 Phân loại chunks:")
for ctype, count in chunk_types.most_common():
    print(f"   {ctype:<20} : {count}")

# ── In toàn bộ Markdown (đầy đủ nhất) ────────────────────────────────────────
print("\n" + "="*60)
print("📝 MARKDOWN TOÀN BỘ FILE (đầu tiên 3000 ký tự):")
print("="*60)
print(doc.markdown[:3000])
print("...\n[xem full bên dưới hoặc lưu file]")

# ── In từng chunk theo trang ──────────────────────────────────────────────────
print("\n" + "="*60)
print("🔍 TỪNG CHUNK THEO LOẠI & TRANG")
print("="*60)

tables_found = []
for i, chunk in enumerate(doc.chunks):
    page_num = chunk.grounding[0].page + 1 if chunk.grounding else "?"
    ctype    = chunk.chunk_type.value

    # In heading và table — 2 loại quan trọng nhất
    if ctype in ("heading", "table", "section_header"):
        print(f"\n[Chunk {i:03d}] Type={ctype} | Trang={page_num}")
        print(chunk.text[:300])
        if ctype == "table":
            tables_found.append({"chunk_id": i, "page": page_num, "text": chunk.text})

print(f"\n\n✅ Tổng bảng phát hiện: {len(tables_found)}")

# ── Kiểm tra từng bảng ────────────────────────────────────────────────────────
print("\n" + "="*60)
print("📊 DANH SÁCH BẢNG TÌM THẤY")
print("="*60)
for t in tables_found:
    first_line = t["text"].split("\n")[0][:80]
    print(f"  Trang {t['page']} | Chunk {t['chunk_id']:03d} | {first_line}")

# ── Lưu kết quả đầy đủ ra file ────────────────────────────────────────────────
with open("sawaco_full_markdown.md", "w", encoding="utf-8") as f:
    f.write(doc.markdown)
print("\n💾 Đã lưu: sawaco_full_markdown.md")

import json
chunks_export = [
    {
        "chunk_id":   i,
        "type":       c.chunk_type.value,
        "page":       c.grounding[0].page + 1 if c.grounding else None,
        "text":       c.text,
        "has_table":  c.chunk_type.value == "table"
    }
    for i, c in enumerate(doc.chunks)
]
with open("sawaco_chunks.json", "w", encoding="utf-8") as f:
    json.dump(chunks_export, f, ensure_ascii=False, indent=2)
print("💾 Đã lưu: sawaco_chunks.json")

In [ ]:
import os
import json
from pathlib import Path
from collections import Counter
from agentic_doc.parse import parse_documents


# ─────────────────────────────────────────────
# CẤU HÌNH
# ─────────────────────────────────────────────
os.environ["VISION_AGENT_API_KEY"] = "aHZkenF6YzFjb3Z6eGNlb2N3cTFjOm83TVY0WngwNTVsUjV4YVBwZ3FwdEJTT0E4eXppN2lO"

# ✏️ THAY ĐỔI PATH NÀY MỖI LẦN CHẠY
TARGET_FOLDER = "/content/drive/MyDrive/LandingAI/bctc_2025/DPM"   # ← chỉnh tay

OUTPUT_DIR = "/content/drive/MyDrive/LandingAI/parsed_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

DONE_LOG = os.path.join(OUTPUT_DIR, "_done.txt")


# ─────────────────────────────────────────────
# HELPERS
# ─────────────────────────────────────────────
def load_done() -> set:
    if not os.path.exists(DONE_LOG):
        return set()
    with open(DONE_LOG, "r", encoding="utf-8") as f:
        return set(line.strip() for line in f if line.strip())


def mark_done(pdf_path: str):
    with open(DONE_LOG, "a", encoding="utf-8") as f:
        f.write(pdf_path + "\n")


def collect_pdfs(folder: str) -> list[str]:
    """Chỉ lấy PDF trong 1 thư mục, KHÔNG đệ quy."""
    return sorted(str(p) for p in Path(folder).glob("*.pdf"))


# ─────────────────────────────────────────────
# PARSE + LƯU 1 FILE
# ─────────────────────────────────────────────
def parse_and_save(pdf_path: str) -> bool:
    pdf_name  = Path(pdf_path).stem
    ticker    = Path(pdf_path).parent.name
    out_subdir = os.path.join(OUTPUT_DIR, ticker)
    os.makedirs(out_subdir, exist_ok=True)

    md_path   = os.path.join(out_subdir, f"{pdf_name}.md")
    json_path = os.path.join(out_subdir, f"{pdf_name}.json")

    if os.path.exists(md_path) and os.path.exists(json_path):
        print(f"  ⏭️  Bỏ qua (đã parse): {ticker}/{pdf_name}.pdf")
        return True

    try:
        print(f"  ⏳ Parsing: {ticker}/{pdf_name}.pdf")
        results = parse_documents([pdf_path])
        doc     = results[0]

        with open(md_path, "w", encoding="utf-8") as f:
            f.write(f"# {pdf_name}\n\n")
            f.write(f"**Ticker:** {ticker}  \n")
            f.write(f"**File:** {pdf_name}.pdf  \n")
            f.write(f"**Trang:** {doc.start_page_idx} → {doc.end_page_idx}  \n")
            f.write(f"**Chunks:** {len(doc.chunks)}  \n\n---\n\n")
            f.write(doc.markdown)

        chunk_types = Counter(c.chunk_type.value for c in doc.chunks)
        tables      = []
        chunks_data = []

        for i, c in enumerate(doc.chunks):
            page  = c.grounding[0].page + 1 if c.grounding else None
            ctype = c.chunk_type.value
            chunks_data.append({
                "chunk_id": i, "type": ctype,
                "page": page,  "text": c.text,
                "is_table": ctype == "table",
            })
            if ctype == "table":
                tables.append({"chunk_id": i, "page": page,
                               "preview": c.text[:120]})

        with open(json_path, "w", encoding="utf-8") as f:
            json.dump({
                "file": pdf_name, "ticker": ticker,
                "doc_type": doc.doc_type,
                "pages": f"{doc.start_page_idx}→{doc.end_page_idx}",
                "total_chunks": len(doc.chunks),
                "chunk_types": dict(chunk_types),
                "tables_count": len(tables),
                "tables": tables, "chunks": chunks_data,
            }, f, ensure_ascii=False, indent=2)

        print(f"  ✅ Done | chunks={len(doc.chunks)} | bảng={len(tables)} | "
              f"trang={doc.start_page_idx}→{doc.end_page_idx}")
        return True

    except Exception as e:
        err = str(e).lower()
        if any(kw in err for kw in ["credit","quota","limit","insufficient",
                                     "balance","payment","402","429"]):
            print(f"\n  💳 HẾT CREDIT! Dừng lại.\n     Chi tiết: {e}")
            return False
        print(f"  ⚠️  Lỗi (bỏ qua): {e}")
        return True


# ─────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────
def run_parse_folder():
    from datetime import datetime
    start    = datetime.now()
    ticker   = Path(TARGET_FOLDER).name
    all_pdfs = collect_pdfs(TARGET_FOLDER)
    done_set = load_done()
    pending  = [p for p in all_pdfs if p not in done_set]

    print("=" * 65)
    print(f"🕐 Bắt đầu   : {start.strftime('%d/%m/%Y %H:%M:%S')}")
    print(f"📁 Thư mục   : {TARGET_FOLDER}")
    print(f"📋 Tổng PDF  : {len(all_pdfs)}")
    print(f"✅ Đã parse  : {len(all_pdfs) - len(pending)}")
    print(f"⏳ Còn lại   : {len(pending)}")
    print("=" * 65 + "\n")

    if not pending:
        print("🎉 Tất cả file trong thư mục đã được parse!")
        return

    success_count = 0
    stopped_at    = None

    for idx, pdf_path in enumerate(pending, 1):
        print(f"\n[{idx:02d}/{len(pending)}] {Path(pdf_path).name}")
        ok = parse_and_save(pdf_path)
        if not ok:
            stopped_at = pdf_path
            break
        mark_done(pdf_path)
        success_count += 1

    elapsed = (datetime.now() - start).seconds
    print(f"\n{'='*65}")
    print(f"📊 KẾT QUẢ — {ticker}")
    print(f"{'='*65}")
    print(f"✅ Thành công : {success_count} file")
    print(f"⏱️  Thời gian  : {elapsed}s")
    if stopped_at:
        remaining = len(pending) - success_count
        print(f"💳 Dừng tại  : {Path(stopped_at).name}")
        print(f"📋 Còn lại   : {remaining} file")
        print(f"👉 Nạp credit xong → chạy lại, tự skip file đã parse")
    else:
        print(f"🎉 Hoàn thành toàn bộ thư mục {ticker}!")
    print(f"📁 Output     : {os.path.join(OUTPUT_DIR, ticker)}")
    print(f"{'='*65}")


run_parse_folder()

🕐 Bắt đầu   : 13/04/2026 19:14:23
📁 Thư mục   : /content/drive/MyDrive/LandingAI/bctc_2025/DPM
📋 Tổng PDF  : 2
✅ Đã parse  : 0
⏳ Còn lại   : 2


[01/2] DPM_2025_Q2_6T_Soatxet_Congtyme.pdf
  ⏳ Parsing: DPM/DPM_2025_Q2_6T_Soatxet_Congtyme.pdf
2026-04-13 19:14:23 [info   ] Parsing 1 documents            [agentic_doc.parse] (parse.py:280)


Parsing documents:   0%|          | 0/1 [00:00<?, ?it/s]

2026-04-13 19:14:24 [info   ] Splitting PDF: '/content/drive/MyDrive/LandingAI/bctc_2025/DPM/DPM_2025_Q2_6T_Soatxet_Congtyme.pdf' into 9 parts under '/tmp/tmpv4bgn2eq' [agentic_doc.utils] (utils.py:238)
2026-04-13 19:14:24 [info   ] Created /tmp/tmpv4bgn2eq/DPM_2025_Q2_6T_Soatxet_Congtyme_1.pdf [agentic_doc.utils] (utils.py:254)
2026-04-13 19:14:24 [info   ] Created /tmp/tmpv4bgn2eq/DPM_2025_Q2_6T_Soatxet_Congtyme_2.pdf [agentic_doc.utils] (utils.py:254)
2026-04-13 19:14:24 [info   ] Created /tmp/tmpv4bgn2eq/DPM_2025_Q2_6T_Soatxet_Congtyme_3.pdf [agentic_doc.utils] (utils.py:254)
2026-04-13 19:14:24 [info   ] Created /tmp/tmpv4bgn2eq/DPM_2025_Q2_6T_Soatxet_Congtyme_4.pdf [agentic_doc.utils] (utils.py:254)
2026-04-13 19:14:24 [info   ] Created /tmp/tmpv4bgn2eq/DPM_2025_Q2_6T_Soatxet_Congtyme_5.pdf [agentic_doc.utils] (utils.py:254)
2026-04-13 19:14:24 [info   ] Created /tmp/tmpv4bgn2eq/DPM_2025_Q2_6T_Soatxet_Congtyme_6.pdf [agentic_doc.utils] (utils.py:254)
2026-04-13 19:14:24 [info   ]


Parsing document parts from 'DPM_2025_Q2_6T_Soatxet_Congtyme.pdf':   0%|          | 0/10 [00:00<?, ?it/s]

2026-04-13 19:14:24 [info   ] Start parsing document part: 'File name: DPM_2025_Q2_6T_Soatxet_Congtyme_4.pdf	Page: [30:39]' [agentic_doc.parse] (parse.py:671)
2026-04-13 19:14:24 [info   ] Start parsing document part: 'File name: DPM_2025_Q2_6T_Soatxet_Congtyme_5.pdf	Page: [40:49]' [agentic_doc.parse] (parse.py:671)


...

HTTP Request: POST https://api.va.landing.ai/v1/tools/agentic-document-analysis "HTTP/1.1 200 OK" (_client.py:1025)
2026-04-13 19:16:36 [info   ] Time taken to successfully parse a document chunk: 39.24 seconds [agentic_doc.parse] (parse.py:827)
2026-04-13 19:16:36 [info   ] Successfully parsed document part: 'File name: DPM_2025_Q2_6T_Soatxet_Hopnhat_4.pdf	Page: [30:39]' [agentic_doc.parse] (parse.py:680)
2026-04-13 19:16:36 [info   ] Start parsing document part: 'File name: DPM_2025_Q2_6T_Soatxet_Hopnhat_6.pdf	Page: [50:59]' [agentic_doc.parse] (parse.py:671)
HTTP Request: POST https://api.va.landing.ai/v1/tools/agentic-document-analysis "HTTP/1.1 200 OK" (_client.py:1025)
2026-04-13 19:16:39 [info   ] Time taken to successfully parse a document chunk: 42.76 seconds [agentic_doc.parse] (parse.py:827)
2026-04-13 19:16:39 [info   ] Successfully parsed document part: 'File name: DPM_2025_Q2_6T_Soatxet_Hopnhat_5.pdf	Page: [40:49]' [agentic_doc.parse] (parse.py:680)
2026-04-13 19:16:39 [i


Parsing document parts from 'DPM_2025_Q2_6T_Soatxet_Hopnhat.pdf':  10%|█         | 1/10 [00:48<07:16, 48.45s/it]

2026-04-13 19:16:45 [info   ] Time taken to successfully parse a document chunk: 48.53 seconds [agentic_doc.parse] (parse.py:827)
2026-04-13 19:16:45 [info   ] Successfully parsed document part: 'File name: DPM_2025_Q2_6T_Soatxet_Hopnhat_2.pdf	Page: [10:19]' [agentic_doc.parse] (parse.py:680)
2026-04-13 19:16:45 [info   ] Start parsing document part: 'File name: DPM_2025_Q2_6T_Soatxet_Hopnhat_10.pdf	Page: [90:92]' [agentic_doc.parse] (parse.py:671)
HTTP Request: POST https://api.va.landing.ai/v1/tools/agentic-document-analysis "HTTP/1.1 402 Payment Required" (_client.py:1025)
2026-04-13 19:16:45 [error  ] Error parsing document 'File name: DPM_2025_Q2_6T_Soatxet_Hopnhat_9.pdf	Page: [80:89]' due to: Client error '402 Payment Required' for url 'https://api.va.landing.ai/v1/tools/agentic-document-analysis'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/402 [agentic_doc.parse] (parse.py:726)
HTTP Request: POST https://api.va.landing.ai/v1/tools/agentic


Parsing document parts from 'DPM_2025_Q2_6T_Soatxet_Hopnhat.pdf': 100%|██████████| 10/10 [01:23<00:00,  8.31s/it]
Parsing documents: 100%|██████████| 1/1 [01:24<00:00, 84.67s/it]

  ✅ Done | chunks=745 | bảng=100 | trang=0→92

📊 KẾT QUẢ — DPM
✅ Thành công : 2 file
⏱️  Thời gian  : 176s
🎉 Hoàn thành toàn bộ thư mục DPM!
📁 Output     : /content/drive/MyDrive/LandingAI/parsed_output/DPM


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive
